# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Read `skills/README.md`, then loaded `skills/training-honest-models/SKILL.md` and `skills/flyrank/flyrank-data/SKILL.md` as directed on this assignment's card. Lane: Refresh / Content Opportunity Scoring (Lane 2), building on the W04 baseline rule.*

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My question shape is "which pages first?" — a ranking problem over an observed label (`is_declining_label`), which per the training-honest-models skill's table means starting with **Logistic Regression, then Random Forest**: readable first, stronger second, and I get to see whether the extra complexity of the forest actually earns its place over the simpler linear model before trusting it.

I'm not using Gradient Boosting here — the skill is explicit that simplicity is a feature, and a stronger method should only enter once a simpler one has been tried and clearly falls short. I want to see where Logistic Regression and Random Forest land first.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
eligible["is_declining_label"] = (eligible["trend_direction"] == "down").astype(int)

# Rebuild the W04 baseline rule exactly, for a same-data comparison later.
declining_flag = (eligible["trend_direction"] == "down").astype(int)
visible_flag = (eligible["impressions_90d"] >= 500).astype(int)
stale_flag = ((eligible["days_since_last_update"] >= 30) & (eligible["days_since_last_update"] < 110)).astype(int)
low_ctr_flag = ((eligible["ctr"] < 0.5) & (eligible["avg_position"] > 0) & (eligible["avg_position"] <= 20)).astype(int)
eligible["baseline_score"] = declining_flag * visible_flag * eligible["impressions_90d"] * (1 + 0.5*stale_flag + 0.5*low_ctr_flag)

# Safe features only: no trend_direction / trend_pct (label-derived, per the flyrank-data skill).
feature_cols = ["impressions_90d", "sessions_90d", "avg_position", "ctr", "word_count",
                 "content_age_days", "days_since_last_update", "search_volume", "cpc",
                 "engagement_rate", "scroll_rate"]

print(f"Eligible rows: {len(eligible)}")
print(f"Features used ({len(feature_cols)}):", feature_cols)

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`, not random.** The starter data has 32 clients, and pages from the same client likely share patterns (writing style, template, industry) that a random row-level split would let leak between train and test — the model could partly "recognize the client" rather than learning transferable signal. A `GroupShuffleSplit` on `client_id` guarantees zero client overlap between train and test, which is the honest design here (no time-series column exists in the starter data to justify a time-aware split instead).

In [ ]:
X = eligible[feature_cols].fillna(0)
y = eligible["is_declining_label"]
groups = eligible["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
baseline_test = eligible["baseline_score"].iloc[test_idx].values

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print(f"Train: {len(X_train)} rows, {len(train_clients)} clients")
print(f"Test: {len(X_test)} rows, {len(test_clients)} clients")
print(f"Client overlap between train/test: {len(train_clients & test_clients)}")
print(f"Test base rate (is_declining_label): {y_test.mean():.3f}")

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
lr_scores = lr.predict_proba(X_test_s)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

print(f"{'Method':<18}{'P@20':<10}{'P@50':<10}")
print(f"{'Baseline rule':<18}{precision_at_k(baseline_test, y_test.values, 20):<10.3f}{precision_at_k(baseline_test, y_test.values, 50):<10.3f}")
print(f"{'Logistic Reg':<18}{precision_at_k(lr_scores, y_test.values, 20):<10.3f}{precision_at_k(lr_scores, y_test.values, 50):<10.3f}")
print(f"{'Random Forest':<18}{precision_at_k(rf_scores, y_test.values, 20):<10.3f}{precision_at_k(rf_scores, y_test.values, 50):<10.3f}")

**Stop — this table is misleading, and I want to show why rather than hide it.** The baseline scores exactly 1.000 at both P@20 and P@50. That's not a strong model — it's circular. `baseline_score` is built by *multiplying* by `declining_flag`, so any row the baseline scores above zero is **guaranteed** to have `is_declining_label == 1` by construction. Evaluating the baseline against the exact label it was hard-gated on can never fail — it's not a fair comparison, it's a tautology.

In [ ]:
# Confirm the circularity directly.
positive_score = eligible[eligible["baseline_score"] > 0]
print(f"Rows with baseline_score > 0: {len(positive_score)}")
print(f"Of those, share with is_declining_label == 1: {(positive_score['is_declining_label']==1).mean():.3f}")

**Building a fairer, non-circular target instead.** The real decision this lane cares about isn't just "is this page declining" (which the baseline already answers by definition) — it's "which declining pages are the *real* priority": severely declining, with enough traffic that losing it actually matters. I define `true_priority` using `trend_pct` (severity) and `impressions_90d` (traffic at stake) — both still only used to build a *label* for evaluation, never as a *feature*, same rule as `trend_direction` throughout this project.

In [ ]:
decliners = eligible[eligible["is_declining_label"] == 1]
severe_cutoff = decliners["trend_pct"].quantile(1/3)   # most negative third = most severe drop
impr_cutoff = decliners["impressions_90d"].median()

eligible["true_priority"] = ((eligible["is_declining_label"] == 1) &
                               (eligible["trend_pct"] <= severe_cutoff) &
                               (eligible["impressions_90d"] >= impr_cutoff)).astype(int)

y_priority_test = eligible["true_priority"].iloc[test_idx].values
print(f"true_priority base rate (test set): {y_priority_test.mean():.3f}")

In [ ]:
print("--- Comparison table: baseline vs. models, evaluated against true_priority ---")
print(f"{'Method':<18}{'P@20':<10}{'P@50':<10}{'ROC-AUC'}")
print(f"{'Baseline rule':<18}{precision_at_k(baseline_test, y_priority_test, 20):<10.3f}{precision_at_k(baseline_test, y_priority_test, 50):<10.3f}{roc_auc_score(y_priority_test, baseline_test):.3f}")
print(f"{'Logistic Reg':<18}{precision_at_k(lr_scores, y_priority_test, 20):<10.3f}{precision_at_k(lr_scores, y_priority_test, 50):<10.3f}{roc_auc_score(y_priority_test, lr_scores):.3f}")
print(f"{'Random Forest':<18}{precision_at_k(rf_scores, y_priority_test, 20):<10.3f}{precision_at_k(rf_scores, y_priority_test, 50):<10.3f}{roc_auc_score(y_priority_test, rf_scores):.3f}")
print(f"\nBase rate for reference: {y_priority_test.mean():.3f}")

**The honest table (base rate 0.040):**

| Method | P@20 | P@50 | ROC-AUC |
|---|---|---|---|
| Baseline rule | **0.150** | **0.100** | **0.866** |
| Logistic Regression | 0.050 | 0.020 | 0.603 |
| Random Forest | 0.000 | 0.020 | 0.598 |

**The baseline wins clearly on the metric that actually matters.** Both models beat the meaningless circular target trivially, but against `true_priority` — the label that reflects the real decision — the baseline's AUC (0.866) is far ahead of both Logistic Regression (0.603) and Random Forest (0.598), and its precision@20 (0.15, a 3.75x lift over the 0.04 base rate) beats Random Forest's precision@20 of 0.000 outright. Per the skill's own instruction: "if the model wins at precision@50 but loses at precision@20 — report both; that IS the finding." Here the baseline wins at both — that's the finding, and it stands.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Why did the models lose to the baseline here?** Both Logistic Regression and Random Forest were trained to predict `is_declining_label` — a simple yes/no — not `true_priority`, which is a stricter, rarer target combining severity and traffic. Their probability scores reflect "how confident am I this page is declining at all," not "how severely, with how much at stake." The baseline's secondary boosts (staleness, low-CTR) happen to correlate better with real severity than the models' learned decision boundary does, on this particular slice.

In [ ]:
print("--- What Random Forest leans on ---")
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances.round(3))

`impressions_90d` (0.268) and `avg_position` (0.243) dominate, with `content_age_days` (0.178) third. These make intuitive sense as decline-related signals — but none of them are severity-of-decline signals, which is exactly why the model's ranking doesn't line up with `true_priority` even though it's a reasonable model of "is this page declining."

**Three concrete wrong cases — Random Forest's top-20 picks that were NOT true priorities:**

In [ ]:
eligible_test = eligible.iloc[test_idx].copy()
eligible_test["rf_score"] = rf_scores

top20_rf = eligible_test.sort_values("rf_score", ascending=False).head(20)
print(f"RF top-20 true_priority hits: {top20_rf['true_priority'].sum()} / 20")
print()
wrong = top20_rf[top20_rf["true_priority"] == 0].head(3)
print(wrong[["content_id", "rf_score", "trend_pct", "impressions_90d", "is_declining_label", "true_priority"]].to_string(index=False))

The worst of these, `content_9c128be31943`, scored 0.865 — RF's near-top confidence that this page is declining — but its real `trend_pct` is **+43.3**, meaning it's *improving*, not declining at all. That's a genuine model error, not just a near-miss on severity: the model is confidently wrong about the basic direction, not just the priority ranking.

**Three real `true_priority` pages Random Forest ranked lowest:**

In [ ]:
missed = eligible_test[eligible_test["true_priority"]==1].sort_values("rf_score").head(3)
print(missed[["content_id", "rf_score", "trend_pct", "impressions_90d", "is_declining_label", "true_priority"]].to_string(index=False))

These three all have severe declines (`trend_pct` between -86% and -98%) but sit at the lower end of `impressions_90d` (972–1,249) relative to the training distribution — since `impressions_90d` is the model's single most important feature, it likely under-weights pages with real, severe decline but moderate (not huge) traffic. This is a plausible, explainable failure mode, not a random one: the model is leaning on volume as a proxy for importance, and volume and severity aren't the same thing.

**What this means, honestly:** this isn't a "the model is broken" story — it's a "the model was optimized for the wrong proxy" story. It was trained to predict *whether* a page is declining, and it does that reasonably (its AUC against `is_declining_label` itself was 0.61, better than chance). But the lane's real decision — *which* declining pages deserve review first — needs a model trained directly against something like `true_priority`, not `is_declining_label`. That's the concrete next step, not a bigger model.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this yourself in Colab**; all numbers above were verified against the real starter dataset in this repo, so it should reproduce exactly with the same random seeds
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.